# Notebook 02 — Stakeholder Analysis

**Study:** Satellite-derived Urban Heat Island Intensity across 56 Target Nigerian Cities  
**Implementation version:** `NB02_v1`  
**Output directory:** `Results/02_stakeholder_analysis/`  
**Depends on NB01:** Loads `city_cluster25_lookup.csv` from `Results/01_aoi_delineation/`.

> **Note:** Stakeholder survey data covers a subset of the 56 study cities.
> Cities with survey coverage are used for hotspot–perception validation in NB05.
> Cities without survey data are not excluded from any LST analysis.

---

<a id="nb02-toc"></a>
## Table of Contents

| Section | Description |
|---------|-------------|
| [§1 — Setup & Imports](#nb02-s01) | Python packages, project paths, config |
| [§2 — Load Required Datasets](#nb02-s02) | Cities CSV, Nigeria boundary, stakeholder survey master composite |
| [§3 — Data Preparation & Verification](#nb02-s03) | Registered stakeholder counts, PID verification, workshop attendance |
| [§4A — Geospatial & Chart Data Preparation](#nb02-s04) | Cities GeoDataFrame, profession colour mapping |
| [§4B — Cluster Data Preparation](#nb02-s04b) | Cluster25 lookup, stakeholder distribution per cluster, cluster profession data |
| [§5A — Stakeholder Distribution Map (City-Level)](#nb02-s05a) | Nigeria base map with proportional circles per city |
| [§5B — Stakeholder Distribution Map (Cluster-Level)](#nb02-s05b) | Nigeria base map with cities coloured by cluster + bar chart |
| [§6 — City & Stakeholder Classifications](#nb02-s06) | Single top-semicircle half-donut: cluster (Ring 1), attendance split (Ring 2), and city arcs (Ring 3) |
| [§7A — Stakeholder Perception Analysis (City-Level)](#nb02-s07a) | Survey Q61 (temperature increase) + Q60 (peak season), rows = cities |
| [§7B — Stakeholder Perception Analysis (Cluster-Level)](#nb02-s07b) | Same survey questions aggregated by Cluster25 |
| [§8 — Export](#nb02-s08) | All figures (SVG + PNG + PDF) + data tables to Results/ |
| [§9 — Summary & Next Steps](#nb02-s09) | Outputs tree; NB05 dependency for perception validation |

---

## Sub-Research Objectives

1. **Map stakeholder geographic distribution** across Nigerian cities with population context
2. **Characterise professional composition** of workshop attendees (academic, practitioner, GIS expert)
3. **Classify city and stakeholder engagement** by registration, attendance, and Cluster25 class
4. **Analyse stakeholder perceptions** on temperature increase and diurnal heat severity
5. **Generate publication-ready visualisations** combining spatial distribution with stakeholder metrics

---

## Workflow

```
Cities CSV + Survey Master Composite
           │
           ▼
  §1 — Setup & Imports
           │
           ▼
  §2 — Load Datasets
           │
           ▼
  §3 — Data Preparation & Verification
           │
           ▼
  §4A — Geospatial Data      §4B — Cluster Data (from NB01)
           │                        │
           └──────────┬─────────────┘
                      ▼
  §5A — City-Level Distribution Map
           │
           ▼
  §5B — Cluster-Level Distribution Map
           │
           ▼
  §6 — City & Stakeholder Classifications (Top-Semicircle Half-Donut)
           │
           ▼
  §7A — City-Level Perception Charts
           │
           ▼
  §7B — Cluster-Level Perception Charts
           │
           ▼
  §8 — Export
```

---

## Data Sources

| Dataset | Source | Purpose |
|---------|--------|---------|
| Cities CSV | UN WUP 2025 | City locations + population |
| `boundary_Nigeria.geojson` | Local | National boundary for map |
| Stakeholder survey (master composite CSV) | Workshop data | Registration, attendance, perceptions |
| `city_cluster25_lookup.csv` | NB01 §11 output | Cluster25 physiographic class per city |

---
<a id="nb02-s01"></a>
## Section 1 — Setup & Imports

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Project root (this notebook lives in /Notebook/) ─────────────────────────
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / 'src'))

import config
from data_io import load_stakeholder_datasets, export_stakeholder_outputs, export_perception_charts
from visualization import (
    create_stakeholder_map_with_bar_chart,
    create_city_stakeholder_donut,
    create_perception_chart_with_donuts,
    create_cluster_perception_chart,
    create_collab_type_chart,
)

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point

# ── Output directory ─────────────────────────────────────────────────────────
RESULTS_STAKE = Path(project_root) / 'Results' / '02_stakeholder_analysis'
RESULTS_STAKE.mkdir(parents=True, exist_ok=True)

print(f"Project root   : {project_root}")
print(f"Output dir     : {RESULTS_STAKE}")
print(f"Study year     : {config.PROJECT_STUDY_YEAR}")
print("Modules loaded : data_io, visualization")

---
<a id="nb02-s02"></a>
## Section 2 — Load Required Datasets

---

> ↑ [Back to Table of Contents](#nb02-toc)

> **Note — cohort definition.** Inclusion is decided by `matched_city_x`, the **registered**
> city, and not by `matched_city_y`, the city of workshop attendance. Registration is the
> auditable record of participation; attendance on its own cannot be verified, because the
> project holds no per-person attendance register. One respondent (`cal235`) marked spots in a
> study city but has no registration entry, so that record is excluded as unverifiable. The
> exclusion is deliberate and conservative, and it fixes the analysed cohort at
> **105 participants**.


In [ ]:
boundary_gdf, cities_df, survey_df = load_stakeholder_datasets(config)

print(f"Cities loaded  : {len(cities_df)}")
print(f"Survey rows    : {len(survey_df)}")
print(f"Unique PIDs    : {survey_df['pid'].nunique()}")

print("\nCities sample:")
print(cities_df[['Urban Agglomeration', 'Latitude', 'Longitude',
                  str(config.PROJECT_STUDY_YEAR)]].head())

print("\nSurvey sample:")
print(survey_df[['pid', 'matched_city_x', 'matched_city_y',
                  '45_disc_prof', 'survey status']].head())

---
<a id="nb02-s03"></a>
## Section 3 — Data Preparation & Verification

Counts registered stakeholders and workshop attendees per city.
Verifies PID uniqueness; multi-city associations are expected and logged.

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
# ── Registered stakeholders per city (matched_city_x) ────────────────────────
registered_counts = (survey_df
    .groupby('matched_city_x')['pid'].nunique()
    .reset_index()
    .rename(columns={'matched_city_x': 'city', 'pid': 'registered_count'}))

# ── Workshop attendees (survey status = Yes, workshop city = WUP study city) ──
# survey_df is already filtered to target-city registrants (matched_city_x).
# Additionally filter matched_city_y to WUP-listed cities so all counts are
# pinned to the same city universe and are fully consistent.
_wup_cities = set(cities_df['Urban Agglomeration'].str.strip())

workshop_df = (survey_df[
        (survey_df['survey status'] == 'Yes') &
        survey_df['matched_city_y'].isin(_wup_cities)
    ].copy())

_n_ws_excl = (survey_df['survey status'] == 'Yes').sum() - len(workshop_df)
if _n_ws_excl:
    _excl = (survey_df[
                 (survey_df['survey status'] == 'Yes') &
                 ~survey_df['matched_city_y'].isin(_wup_cities)
             ]['matched_city_y'].value_counts())
    print(f"ℹ  {_n_ws_excl} workshop record(s) with non-WUP city excluded from analysis:")
    for city, n in _excl.items():
        print(f"   {city}: {n} record(s)")

workshop_prof = (workshop_df
    .groupby(['matched_city_y', '45_disc_prof'])['pid'].nunique()
    .reset_index()
    .rename(columns={'matched_city_y': 'city', '45_disc_prof': 'profession', 'pid': 'count'}))

workshop_total = (workshop_df
    .groupby('matched_city_y')['pid'].nunique()
    .reset_index()
    .rename(columns={'matched_city_y': 'city', 'pid': 'total_attendees'}))

print(f"Registered stakeholders : {survey_df['pid'].nunique()} unique PIDs, "
      f"{len(registered_counts)} cities")
print(f"Workshop attendees      : {workshop_df['pid'].nunique()} unique PIDs, "
      f"{len(workshop_total)} cities")
print(f"Profession categories   : {workshop_prof['profession'].nunique()}")

print("\nTop 10 cities by registered stakeholders:")
print(registered_counts.sort_values('registered_count', ascending=False)
      .head(10).to_string(index=False))

In [ ]:
# ── PID verification ──────────────────────────────────────────────────────────
pid_cities_x = survey_df.groupby('pid')['matched_city_x'].nunique()
pid_cities_y = survey_df.groupby('pid')['matched_city_y'].nunique()
multi_x = pid_cities_x[pid_cities_x > 1]
multi_y = pid_cities_y[pid_cities_y > 1]

print("Registration (matched_city_x):")
if len(multi_x):
    print(f"  {len(multi_x)} PIDs associated with multiple cities (expected)")
else:
    print("  All PIDs unique to single city")

print("Workshop attendance (matched_city_y):")
if len(multi_y):
    print(f"  {len(multi_y)} PIDs attended workshops in multiple cities")
    for pid in multi_y.head(3).index:
        cities = survey_df[survey_df['pid'] == pid]['matched_city_y'].unique()
        print(f"    PID {pid}: {', '.join(cities)}")
else:
    print("  All PIDs attended workshop in single city")

print(f"\nTotal unique PIDs           : {survey_df['pid'].nunique()}")
print(f"Total unique workshop PIDs  : {workshop_df['pid'].nunique()}")
print(f"Sum registered counts       : {registered_counts['registered_count'].sum()}")

---
<a id="nb02-s04"></a>
## Section 4A — Geospatial & Chart Data Preparation

Builds the cities GeoDataFrame, merges stakeholder counts, and sets up profession colour mapping.

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
# ── Cities GeoDataFrame ───────────────────────────────────────────────────────
cities_geo = cities_df[['Urban Agglomeration', 'Latitude', 'Longitude',
                          str(config.PROJECT_STUDY_YEAR)]].copy()
cities_geo.columns = ['city', 'latitude', 'longitude', 'population']
cities_geo = cities_geo.dropna(subset=['latitude', 'longitude'])

cities_gdf = gpd.GeoDataFrame(
    cities_geo,
    geometry=[Point(xy) for xy in zip(cities_geo['longitude'], cities_geo['latitude'])],
    crs=config.WGS84,
)

# Merge stakeholder counts
cities_gdf = (cities_gdf
    .merge(registered_counts, on='city', how='left')
    .merge(workshop_total,    on='city', how='left'))
cities_gdf['registered_count'] = cities_gdf['registered_count'].fillna(0).astype(int)
cities_gdf['total_attendees']  = cities_gdf['total_attendees'].fillna(0).astype(int)

print(f"Cities GeoDataFrame      : {len(cities_gdf)} cities")
print(f"With registered data     : {(cities_gdf['registered_count'] > 0).sum()}")
print(f"With workshop attendees  : {(cities_gdf['total_attendees'] > 0).sum()}")

print("\nCities with stakeholder data:")
stake_cities = (cities_gdf[cities_gdf['registered_count'] > 0]
    .sort_values('registered_count', ascending=False))
print(stake_cities[['city', 'population', 'registered_count', 'total_attendees']]
    .to_string(index=False))

---
<a id="nb02-s04b"></a>
## Section 4B — Cluster Data Preparation

Loads the NB01 Cluster25 lookup, builds the extended city→cluster map (study + absorbed cities),
merges cluster assignments into `cities_gdf`, and computes the stakeholder distribution table
per cluster and the cluster-level profession breakdown.

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
import re as _re

# ── Load NB01 cluster lookup (56 study cities) ────────────────────────────────
_lkup_path = project_root / 'Results/01_aoi_delineation/city_cluster25_lookup.csv'
if not _lkup_path.exists():
    raise FileNotFoundError(
        f'Cluster lookup not found: {_lkup_path}\n'
        'Run NB01 Section 11 first to generate city_cluster25_lookup.csv'
    )
_cluster_df = pd.read_csv(_lkup_path)
print(f'Cluster lookup loaded : {len(_cluster_df)} study cities, '
      f'{_cluster_df["Cluster25"].nunique()} clusters')

# ── Absorbed cities → parent cluster lookup ───────────────────────────────────
_absorbed_path = project_root / 'Results/01_aoi_delineation/absorbed_cities.csv'
_absorbed_df   = pd.read_csv(_absorbed_path)
_parent_lu = _cluster_df[['city_name', 'Cluster25']].rename(columns={'city_name': 'parent_aoi'})
_absorbed_lu = (
    _absorbed_df.merge(_parent_lu, on='parent_aoi', how='left')
    [['absorbed_city', 'Cluster25']].rename(columns={'absorbed_city': 'city_name'})
)
# Full city → cluster lookup (study + absorbed cities)
_full_city_lu = (
    pd.concat([_cluster_df[['city_name', 'Cluster25']], _absorbed_lu])
    .drop_duplicates('city_name').reset_index(drop=True)
)
print(f'Extended city lookup  : {len(_full_city_lu)} cities (study + absorbed)')

# ── Elevation label formatter ──────────────────────────────────────────────────
def _elev_fmt(elev_class):
    s = str(elev_class).strip()
    m = _re.match(r'^(.*?)\s*(\(.*\))$', s)
    return f"{m.group(1).strip()} elevation {m.group(2)}" if m else f"{s} elevation"

# ── Cluster metadata ───────────────────────────────────────────────────────────
_cmeta_nb02 = (
    _cluster_df.drop_duplicates('Cluster25').copy()
    .assign(
        cluster_label=lambda df: df['Cluster25'].apply(lambda x: f'C{int(x)}'),
        description=lambda df: df.apply(
            lambda r: (
                f"{r['Climate_Class']}, {r['Geology_Class']}, "
                f"and {_elev_fmt(r['Elevation_Class']).lower()}"
            ), axis=1
        )
    )
    [['Cluster25', 'cluster_label', 'Climate_Class', 'Geology_Class',
      'Elevation_Class', 'description']]
    .sort_values('Cluster25').reset_index(drop=True)
)

# ── Merge Cluster25 into cities_gdf ───────────────────────────────────────────
if 'Cluster25' not in cities_gdf.columns:
    cities_gdf = cities_gdf.merge(
        _cluster_df[['city_name', 'Cluster25']].rename(columns={'city_name': 'city'}),
        on='city', how='left'
    )
cities_gdf['cluster_label'] = (
    cities_gdf['Cluster25']
    .apply(lambda x: f'C{int(x)}' if pd.notna(x) else None)
)
_study_gdf = cities_gdf[cities_gdf['Cluster25'].notna()].copy()
print(f'Study cities in cities_gdf        : {len(_study_gdf)}/56')
print(f'Non-study (absorbed/off-study)     : {cities_gdf["Cluster25"].isna().sum()}')

In [ ]:
from IPython.display import display

# ── Stakeholder distribution per cluster — 56 study cities only ───────────────
# Cities with no stakeholders appear as zeros (not excluded).
_total_reg = int(_study_gdf['registered_count'].sum())
_total_att = int(_study_gdf['total_attendees'].sum())

_rows = []
for _cid, _grp in _study_gdf.groupby('cluster_label', sort=True):
    _reg  = int(_grp['registered_count'].sum())
    _att  = int(_grp['total_attendees'].sum())
    _n_w  = int((_grp['registered_count'] > 0).sum())
    _desc = (
        _cmeta_nb02
        .loc[_cmeta_nb02['cluster_label'] == _cid, 'description']
        .iloc[0]
        if not _cmeta_nb02[_cmeta_nb02['cluster_label'] == _cid].empty
        else ''
    )
    _rows.append({
        'Cluster':                  _cid,
        'N Cities':                 len(_grp),
        'Cities w/ Stakeholders':   _n_w,
        'Cities w/o Stakeholders':  len(_grp) - _n_w,
        'Registered':               _reg,
        'Attended Workshop':        _att,
        '% of Registered':          round(_reg / _total_reg * 100, 1) if _total_reg else 0.0,
        '% of Attended':            round(_att / _total_att * 100, 1) if _total_att else 0.0,
        'Description':              _desc,
    })

cluster_stake_df = pd.DataFrame(_rows).reset_index(drop=True)

# cluster_summary format needed by create_cluster_stakeholder_map (Cluster25 as int)
_cluster_summary_map = cluster_stake_df.copy()
_cluster_summary_map['Cluster25'] = _cluster_summary_map['Cluster'].str.lstrip('C').astype(int)

print(f'Clusters : {len(cluster_stake_df)}  |  '
      f'Total registered: {_total_reg}  |  Total attended: {_total_att}')
try:
    display(
        cluster_stake_df.style
        .set_caption(
            f'Table: Stakeholder Distribution per Physiographic Cluster '
            f'(56 study cities, {len(cluster_stake_df)} clusters)'
        )
        .set_table_styles([
            {'selector': 'th',
             'props': [('background-color', '#404040'), ('color', 'white'),
                       ('font-weight', 'bold')]},
            {'selector': 'caption',
             'props': [('font-weight', 'bold'), ('font-size', '13px'),
                       ('text-align', 'left')]},
        ])
        .set_properties(**{'font-size': '11px'})
        .set_properties(
            subset=['Description'],
            **{'white-space': 'normal', 'text-align': 'left', 'max-width': '380px'}
        )
        .set_properties(
            subset=['Cluster', 'N Cities', 'Cities w/ Stakeholders',
                    'Cities w/o Stakeholders'],
            **{'text-align': 'center'}
        )
        .bar(subset=['Registered'],        color='#4472C4', vmin=0)
        .bar(subset=['Attended Workshop'],  color='#ED7D31', vmin=0)
        .format({'% of Registered': '{:.1f}%', '% of Attended': '{:.1f}%'})
    )
except Exception:
    print(cluster_stake_df.to_string(index=False))

# ── Save distribution table ────────────────────────────────────────────────────
_tbl_path = RESULTS_STAKE / 'cluster25_stakeholder_distribution.csv'
cluster_stake_df.to_csv(_tbl_path, index=False)
print(f'Saved: {_tbl_path.name}  ({len(cluster_stake_df)} clusters)')

In [ ]:
# ── Profession colour mapping ─────────────────────────────────────────────────
all_professions = sorted(workshop_prof['profession'].unique())

profession_colors = {
    'Academic':                                        '#1f77b4',
    'Practitioner (Public/private)':                   '#ff7f0e',
    'GIS expert':                                      '#2ca02c',
    'Academic|Practitioner (Public/private)':          '#d62728',
    'GIS expert|Practitioner (Public/private)':        '#9467bd',
    'Academic|GIS expert':                             '#8c564b',
    'Academic|GIS expert|Practitioner (Public/private)': '#e377c2',
}
fallback = ['#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78', '#98df8a']
for i, prof in enumerate([p for p in all_professions if p not in profession_colors
                           and pd.notna(p)]):
    profession_colors[prof] = fallback[i % len(fallback)]

# Profession data dict: {city: {profession: count}}
profession_data = {
    city: dict(zip(d['profession'], d['count']))
    for city, d in workshop_prof.groupby('city')
}

print(f"Profession categories : {len(all_professions)}")
prof_summary = workshop_prof.groupby('profession')['count'].sum().sort_values(ascending=False)
print(prof_summary.to_string())

In [ ]:
# ── Cluster-level profession data for §5B map ─────────────────────────────────
# workshop_df already carries Cluster25 (registration-city cluster from load_stakeholder_datasets).
# We want the WORKSHOP-CITY cluster from _full_city_lu, so rename it before merging
# to prevent pandas creating Cluster25_x / Cluster25_y suffixes.
_full_city_lu_ws = _full_city_lu.rename(columns={
    'city_name': 'matched_city_y',
    'Cluster25': 'Cluster25_ws',
})
_workshop_cluster = workshop_df.merge(_full_city_lu_ws, on='matched_city_y', how='left')

cluster_workshop_prof = (
    _workshop_cluster[_workshop_cluster['Cluster25_ws'].notna()]
    .groupby(['Cluster25_ws', '45_disc_prof'])['pid'].nunique()
    .reset_index()
    .rename(columns={'Cluster25_ws': 'Cluster25', '45_disc_prof': 'profession', 'pid': 'count'})
)
cluster_workshop_prof['Cluster25'] = cluster_workshop_prof['Cluster25'].astype(int)

print(f"Cluster workshop profession records : {len(cluster_workshop_prof)}")
print("Attendees per cluster (top 10):")
print(cluster_workshop_prof.groupby('Cluster25')['count'].sum()
      .sort_values(ascending=False).head(10).to_string())

---
<a id="nb02-s05a"></a>
## Section 5A — Stakeholder Distribution Map (City-Level)

Nigeria base map with:
- Proportional circles sized by registered stakeholder count
- Stacked bar insets per city showing profession breakdown
- National boundary overlay

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
global_registered = survey_df['pid'].nunique()
global_attendees  = workshop_df['pid'].nunique()

# Backward-compatible population column expected by some plotting utilities
pop_col = f"population_{config.PROJECT_STUDY_YEAR}"
if pop_col not in cities_gdf.columns and 'population' in cities_gdf.columns:
    cities_gdf[pop_col] = cities_gdf['population']

stakeholder_map = create_stakeholder_map_with_bar_chart(
    cities_gdf=cities_gdf,
    boundary_gdf=boundary_gdf,
    workshop_prof=workshop_prof,
    profession_colors=profession_colors,
    survey_df=survey_df,
    workshop_total=workshop_total,
    global_registered=global_registered,
    global_attendees=global_attendees,
    figsize=(11.7, 5.85),
)
plt.show()
print("Stakeholder distribution map created.")

---
<a id="nb02-s05b"></a>
## Section 5B — Stakeholder Distribution Map (Cluster-Level)

Single combined figure:
- **Panel A (left):** Same Nigeria proportional-circle map as §5A — circles sized by registered stakeholder count, pie-split by attendance status
- **Panel B (right):** Stacked horizontal bar chart — workshop attendees per Cluster25 class, broken down by profession (consistent `profession_colors`)

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
# ── §5B — Cluster-level stakeholder distribution (single combined figure) ─────
# Panel A (left) : Same Nigeria proportional-circle map as §5A
# Panel B (right): Workshop attendees by profession per cluster (cluster bar chart)
# Passing cluster_workshop_prof triggers the cluster bar panel instead of city bars.

fig_cluster_map = create_stakeholder_map_with_bar_chart(
    cities_gdf=cities_gdf,
    boundary_gdf=boundary_gdf,
    workshop_prof=workshop_prof,
    profession_colors=profession_colors,
    survey_df=survey_df,
    workshop_total=workshop_total,
    global_registered=global_registered,
    global_attendees=global_attendees,
    cluster_workshop_prof=cluster_workshop_prof,
    figsize=(11.7, 5.85),
)
plt.show()
print("Section 5B map created.")

---
<a id="nb02-s06"></a>
## Section 6 — City & Stakeholder Classification

Single top-semicircle half-donut chart.

| Element | Content | Unit |
|---------|---------|------|
| **Center (top)** | Total target cities (NB01, dynamic) | N cities |
| **Center (bottom)** | Unique stakeholders with survey "Yes" (incl. absorbed cities) | N stakeholders |
| **Ring 1 (inner)** | All 16 Cluster25 classes — clusters with any attended cities appear first | N cities per cluster |
| **Ring 2 (middle)** | Workshop attendance split within each cluster — attended (cluster colour) / not attended (lightened cluster colour) | N cities |
| **Ring 3 (outer)** | All 56 cities, arcs ∝ attendees for attended / equal arc for not-attended, colour graded per cluster | N attendees |
| **Outer labels** | Attended cities only — "City Name (N attendees)"; not-attended cities carry no label | — |

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
# ── §6 — City & Stakeholder Classification (Single Half-Donut) ────────────────
# Top-half orientation: arc spans 0° (right) → 90° (top) → 180° (left).
# Ring 1  : 16 clusters — attended-clusters first (sorted)
# Ring 2  : Workshop attendance split per cluster (attended | not-attended)
# Ring 3  : 56 cities — labels only for attended cities, right→right / left→left

fig_cluster_donuts = create_city_stakeholder_donut(
    cluster_summary=cluster_stake_df,
    study_gdf=_study_gdf,
    cluster25_colors=config.CLUSTER25_COLORS,
    total_pids=survey_df['pid'].nunique(),
    attended_pids=workshop_df['pid'].nunique(),
    figsize=(16, 9),
)

# ── Save §6 figure as SVG and PNG ────────────────────────────────────────────
_figs_dir = RESULTS_STAKE / 'Figures'
_figs_dir.mkdir(parents=True, exist_ok=True)

# Save the half-donut figure
_fname = 'city_stakeholder_classification_donut'
for _ext in ['.svg', '.png']:
    _p = _figs_dir / f'{_fname}{_ext}'
    fig_cluster_donuts.savefig(_p, bbox_inches='tight', dpi=config.FIGURE_DPI)
    print(f'✓ Saved: {_p.name}')

print(f'\nFigure saved to: {_figs_dir}')

plt.show()
print(f"Half-donut: {int(cluster_stake_df['N Cities'].sum())} target cities  |  "
      f"{workshop_df['pid'].nunique()} survey-attended stakeholders")

---
<a id="nb02-s06b"></a>
## Section 6b — Collaboration Type Combinations (Registered Stakeholders)

Visualises how the **321 registered stakeholders** are distributed across the unique
self-selected collaboration-type **combinations** from the registration form
(`9_reg_collab_type`, pipe-separated field).

Each respondent is counted **exactly once** under their full combination string
(e.g. `Individual | Research Group`). No exploding — one bar segment per unique combination.

| Chart | Detail |
|-------|--------|
| **City chart** | Horizontal stacked bar per city (33 cities), sorted by total registrants |
| **Cluster chart** | Same aggregated by Cluster25 registration-city class |
| **Segments** | Each unique combination → one colour → one legend entry |
| **Total labels** | Unique PIDs per bar (sum = registered stakeholders with collab data) |

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
# ── §6b Data Preparation: unique collaboration-type combinations ──────────────
# Unit: unique PIDs per (city, combination).
# Each PID gets ONE canonical combination label; multi-city registrants appear
# in every city they joined — consistent with §3 registered_counts totals.
# "All collaboration types" replaces the full six-type combination string.

def _make_combo_label(raw_str):
    parts = sorted(p.strip() for p in str(raw_str).split('|') if p.strip())
    return 'All collaboration types' if len(parts) == 6 else ' | '.join(parts)

# ── Step 1: canonical combo per PID ──────────────────────────────────────────
_collab_with_type = survey_df[survey_df['9_reg_collab_type'].notna()]
_pid_combo = (
    _collab_with_type
    .drop_duplicates('pid')[['pid', '9_reg_collab_type']]
    .copy()
)
_pid_combo['collab_combo'] = _pid_combo['9_reg_collab_type'].apply(_make_combo_label)

# ── Step 2: attach combo to all city rows ─────────────────────────────────────
_collab_data = _collab_with_type.merge(
    _pid_combo[['pid', 'collab_combo']], on='pid', how='inner'
)

# ── Step 3: unique combinations, ordered by global frequency ──────────────────
_combo_freq = _pid_combo['collab_combo'].value_counts()
COMBO_TYPES = _combo_freq.index.tolist()
_n_combos   = len(COMBO_TYPES)

# One distinct colour per combination — chain tab20 + tab20b + tab20c (≤60)
_all_palette = []
for _cm_name in ['tab20', 'tab20b', 'tab20c']:
    _cm = plt.get_cmap(_cm_name)
    _all_palette += [_cm(i / 19) for i in range(20)]
COMBO_COLORS = {c: _all_palette[i] for i, c in enumerate(COMBO_TYPES)}

# ── Step 4: per-city pivot ─────────────────────────────────────────────────────
_city_cc = (
    _collab_data
    .groupby(['matched_city_x', 'collab_combo'])['pid']
    .nunique()
    .reset_index()
    .rename(columns={'matched_city_x': 'city', 'pid': 'count'})
)
city_combo_pivot = (
    _city_cc
    .pivot_table(index='city', columns='collab_combo',
                 values='count', fill_value=0)
    .reindex(columns=COMBO_TYPES, fill_value=0)
)
city_combo_pivot['_total'] = city_combo_pivot[COMBO_TYPES].sum(axis=1)

# ── Step 5: per-cluster pivot (registration-city Cluster25) ───────────────────
_city_cl_lu2 = _cluster_df[['city_name', 'Cluster25']].rename(columns={'city_name': 'city'})
_city_cc_cl  = _city_cc.merge(_city_cl_lu2, on='city', how='left')
_city_cc_cl  = _city_cc_cl[_city_cc_cl['Cluster25'].notna()].copy()
_city_cc_cl['cluster_label'] = _city_cc_cl['Cluster25'].apply(lambda x: f'C{int(x)}')

cluster_combo_pivot = (
    _city_cc_cl
    .groupby(['cluster_label', 'collab_combo'])['count']
    .sum()
    .reset_index()
    .pivot_table(index='cluster_label', columns='collab_combo',
                 values='count', fill_value=0)
    .reindex(columns=COMBO_TYPES, fill_value=0)
)
cluster_combo_pivot['_total'] = cluster_combo_pivot[COMBO_TYPES].sum(axis=1)

# ── Step 6: y-axis label dictionaries ─────────────────────────────────────────
# City chart: "Lagos (Cl-13)"
_city_cl_lookup = dict(zip(_cluster_df['city_name'], _cluster_df['Cluster25'].astype(int)))
_city_combo_ylabel = {
    city: (f'{city} (Cl-{_city_cl_lookup[city]})' if city in _city_cl_lookup else city)
    for city in city_combo_pivot.index
}

# Cluster chart: "Cl-13 (14 cities)"
_clust_n_map = dict(zip(cluster_stake_df['Cluster'], cluster_stake_df['N Cities']))
_clust_combo_ylabel = {
    cl: f'Cl-{cl[1:]} ({_clust_n_map.get(cl, "?")} cities)'
    for cl in cluster_combo_pivot.index
}

# ── Summary ────────────────────────────────────────────────────────────────────
_n_pids_with_data = _pid_combo['pid'].nunique()
_total_city_sum   = int(city_combo_pivot['_total'].sum())
print(f'Unique PIDs with collab data : {_n_pids_with_data}')
print(f'Unique combinations          : {_n_combos}')
print(f'Total city-pivot sum         : {_total_city_sum}  '
      f'(multi-city registrants appear in each city)')
print(f'Cities in pivot              : {len(city_combo_pivot)}')
print(f'Clusters in pivot            : {len(cluster_combo_pivot)}')
print()
print('Unique combination frequencies:')
for combo, cnt in _combo_freq.items():
    print(f'  {combo:<70} {cnt:>4}  ({100*cnt/_n_pids_with_data:.1f}%)')

In [ ]:
# ── §6b Chart A — City-level (no legend; see shared legend below) ─────────────
_ykeys       = list(city_combo_pivot.sort_values('_total', ascending=True).index)
_city_sorted = city_combo_pivot.loc[_ykeys]
_ylabels     = [_city_combo_ylabel.get(c, c) for c in _ykeys]
_active_cols = [c for c in COMBO_TYPES if _city_sorted[c].sum() > 0]
_n_cities    = len(_city_sorted)

fig_combo_city, ax = plt.subplots(figsize=(13, max(5, _n_cities * 0.40)))

left = np.zeros(_n_cities)
for col in _active_cols:
    vals = _city_sorted[col].values
    ax.barh(range(_n_cities), vals, left=left,
            color=COMBO_COLORS[col], edgecolor='white', linewidth=0.5)
    left += vals

for i, city in enumerate(_ykeys):
    total = int(_city_sorted.loc[city, '_total'])
    if total > 0:
        ax.text(total + 0.2, i, str(total),
                va='center', ha='left', fontsize=8, color='#333333')

ax.set_yticks(range(_n_cities))
ax.set_yticklabels(_ylabels, fontsize=9)
ax.set_xlabel('Registered stakeholders (unique PIDs per city)', fontsize=10)
ax.set_title('Collaboration type combinations — City level', fontsize=12, fontweight='bold', pad=8)
ax.set_xlim(0, _city_sorted['_total'].max() * 1.08)
ax.tick_params(axis='x', labelsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
print('City-level combination chart created.')

In [ ]:
# ── §6b Chart B — Cluster-level (no legend; see shared legend below) ──────────
_ckeys         = list(cluster_combo_pivot.sort_values('_total', ascending=True).index)
_clust_sorted  = cluster_combo_pivot.loc[_ckeys]
_clabels       = [_clust_combo_ylabel.get(c, c) for c in _ckeys]
_active_cols_c = [c for c in COMBO_TYPES if _clust_sorted[c].sum() > 0]
_n_clusters    = len(_clust_sorted)

fig_combo_cluster, ax = plt.subplots(figsize=(13, max(4, _n_clusters * 0.52)))

left = np.zeros(_n_clusters)
for col in _active_cols_c:
    vals = _clust_sorted[col].values
    ax.barh(range(_n_clusters), vals, left=left,
            color=COMBO_COLORS[col], edgecolor='white', linewidth=0.5)
    left += vals

for i, cl in enumerate(_ckeys):
    total = int(_clust_sorted.loc[cl, '_total'])
    if total > 0:
        ax.text(total + 0.15, i, str(total),
                va='center', ha='left', fontsize=8, color='#333333')

ax.set_yticks(range(_n_clusters))
ax.set_yticklabels(_clabels, fontsize=9)
ax.set_xlabel('Registered stakeholders (unique PIDs per cluster)', fontsize=10)
ax.set_title('Collaboration type combinations — Cluster level', fontsize=12, fontweight='bold', pad=8)
ax.set_xlim(0, _clust_sorted['_total'].max() * 1.08)
ax.tick_params(axis='x', labelsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
print('Cluster-level combination chart created.')

In [ ]:
# ── §6b Shared Legend (separate figure — applies to both charts above) ────────
from matplotlib.patches import Patch

# Only show combinations that appear in at least one of the two charts
_legend_combos = [c for c in COMBO_TYPES
                  if city_combo_pivot[c].sum() > 0 or cluster_combo_pivot[c].sum() > 0]

_n_leg = len(_legend_combos)
fig_combo_legend, ax_leg = plt.subplots(figsize=(8.5, max(2.5, _n_leg * 0.35)))
ax_leg.set_axis_off()

handles = [
    Patch(facecolor=COMBO_COLORS[c], edgecolor='#888888', linewidth=0.6, label=c)
    for c in _legend_combos
]
ax_leg.legend(
    handles=handles,
    title='Collaboration type combination',
    loc='center',
    fontsize=8,
    title_fontsize=10,
    framealpha=0.95,
    edgecolor='#cccccc',
    ncol=1,
)
fig_combo_legend.suptitle(
    'Legend — shared across city and cluster charts',
    fontsize=9, color='#555555', y=0.98,
)
plt.tight_layout()
plt.show()
print(f'Shared legend created: {_n_leg} combination types.')

In [ ]:
# ── §6b Export ────────────────────────────────────────────────────────────────
_figs_dir = RESULTS_STAKE / 'Figures'
_data_dir = RESULTS_STAKE / 'Data'
_figs_dir.mkdir(parents=True, exist_ok=True)
_data_dir.mkdir(parents=True, exist_ok=True)

for _ext in ['.svg', '.pdf', '.png']:
    for _fname, _fig in [
        ('collab_combo_city_chart',    fig_combo_city),
        ('collab_combo_cluster_chart', fig_combo_cluster),
        ('collab_combo_legend',        fig_combo_legend),
    ]:
        _p = _figs_dir / f'{_fname}{_ext}'
        _fig.savefig(_p, bbox_inches='tight', dpi=config.FIGURE_DPI)
        print(f'  {_p.name}')

city_combo_pivot.drop(columns=['_total']).to_csv(
    _data_dir / 'collab_combo_by_city.csv')
cluster_combo_pivot.drop(columns=['_total']).to_csv(
    _data_dir / 'collab_combo_by_cluster.csv')
print('  collab_combo_by_city.csv  saved')
print('  collab_combo_by_cluster.csv  saved')

---
<a id="nb02-s07a"></a>
## Section 7A — Stakeholder Perception Analysis (City-Level)

Two survey questions:
- **Q61:** Has the city warmed noticeably in the past 10 years?
- **Q60:** When is diurnal heat severity worst?

Outputs: combined stacked-bar chart with embedded donut summaries, rows = cities.

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
# One response per participant (first occurrence per PID)
unique_resp = workshop_df.groupby('pid').agg(
    city=('matched_city_y', 'first'),
    warming=('61_temp_increase_10yrs', 'first'),
    diurnal=('60_diurnal_severity_heat', 'first'),
).reset_index()

unique_resp['warming'] = unique_resp['warming'].fillna("I'd rather not say")
unique_resp['diurnal'] = unique_resp['diurnal'].fillna("I'd rather not say")

warming_total = unique_resp['warming'].value_counts()
diurnal_total = unique_resp['diurnal'].value_counts()

print(f"Unique participants analysed : {len(unique_resp)}")
print(f"Cities represented          : {unique_resp['city'].nunique()}")
print("\nTemperature increase perception:")
print(warming_total.to_string())
print("\nDiurnal heating severity:")
print(diurnal_total.to_string())

In [ ]:
# The plotting helper expects original survey column names plus matched_city_y
unique_resp_plot = unique_resp.copy()

if 'matched_city_y' not in unique_resp_plot.columns and 'city' in unique_resp_plot.columns:
    unique_resp_plot['matched_city_y'] = unique_resp_plot['city']

if '61_temp_increase_10yrs' not in unique_resp_plot.columns and 'warming' in unique_resp_plot.columns:
    unique_resp_plot['61_temp_increase_10yrs'] = unique_resp_plot['warming']

if '60_diurnal_severity_heat' not in unique_resp_plot.columns and 'diurnal' in unique_resp_plot.columns:
    unique_resp_plot['60_diurnal_severity_heat'] = unique_resp_plot['diurnal']

perception_fig, perception_donuts_fig = create_perception_chart_with_donuts(
    unique_resp=unique_resp_plot,
    warming_total=warming_total,
    diurnal_total=diurnal_total,
    min_participants=2,
    figsize=(14, 5.5),
    legend_scale=1.2,
)
plt.show()
print("Perception charts created.")

---
<a id="nb02-s07b"></a>
## Section 7B — Stakeholder Perception Analysis (Cluster-Level)

Aggregates the same Q61 (city warming) and Q60 (diurnal heat severity) responses
by **Cluster25 physiographic class** instead of individual cities.

Off-target participants (no Cluster25 assignment) are excluded and reported in the printout.

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
# ── §7B — Cluster-level perception charts ─────────────────────────────────────
# Mirrors §7A but rows = Cluster25 classes.
# Uses _full_city_lu (study + absorbed → Cluster25) defined in §4B.

# ── Map workshop responses to Cluster25 ───────────────────────────────────────
_prc = unique_resp.merge(
    _full_city_lu.rename(columns={'city_name': 'city'}),
    on='city', how='left'
)
_prc['cluster_label'] = (
    _prc['Cluster25'].apply(lambda x: f'C{int(x)}' if pd.notna(x) else 'C*')
)

# Report + exclude off-target (no Cluster25 assignment)
_n_raw      = len(_prc)
_cstar_rows = _prc[_prc['cluster_label'] == 'C*']
_n_cstar    = len(_cstar_rows)
if _n_cstar > 0:
    _cstar_cities = sorted(_cstar_rows['city'].dropna().unique().tolist())
    print(f"ℹ  Off-target participants excluded from cluster analysis : {_n_cstar}")
    print(f"   Cities (not in 56-city study scope) : {', '.join(_cstar_cities)}")

_prc = _prc[_prc['cluster_label'] != 'C*'].copy()
_n_total = len(_prc)
print(f"Participants in cluster perception analysis : {_n_total}  (of {_n_raw} total)")

# ── Crosstabs ──────────────────────────────────────────────────────────────────
_cl_counts    = _prc.groupby('Cluster25').size().reset_index(name='n')
_cluster_order = _cl_counts.sort_values('n', ascending=False)['Cluster25'].tolist()

_prc_plt = _prc.copy()
_prc_plt['61_temp_increase_10yrs']   = _prc_plt['warming']
_prc_plt['60_diurnal_severity_heat'] = _prc_plt['diurnal']

_w_ctab = (pd.crosstab(_prc_plt['Cluster25'], _prc_plt['61_temp_increase_10yrs'])
           .reindex(_cluster_order).fillna(0))
_d_ctab = (pd.crosstab(_prc_plt['Cluster25'], _prc_plt['60_diurnal_severity_heat'])
           .reindex(_cluster_order).fillna(0))

_w_total_c = _prc_plt['61_temp_increase_10yrs'].value_counts()
_d_total_c = _prc_plt['60_diurnal_severity_heat'].value_counts()

# ── Plot ───────────────────────────────────────────────────────────────────────
fig_prc_clust, _prc_donuts = create_cluster_perception_chart(
    warming_by_cluster=_w_ctab,
    diurnal_by_cluster=_d_ctab,
    warming_total=_w_total_c,
    diurnal_total=_d_total_c,
    cluster25_colors=config.CLUSTER25_COLORS,
    figsize=(14, 5.5),
    legend_scale=1.2,
)
plt.show()
print(f'Cluster perception chart : {len(_cluster_order)} clusters, '
      f'{_n_total} on-target participants')

---
<a id="nb02-s08"></a>
## Section 8 — Export

Saves all figures (SVG + PNG + PDF) and data tables to `Results/02_stakeholder_analysis/`.

---

> ↑ [Back to Table of Contents](#nb02-toc)

In [ ]:
import importlib, data_io
importlib.reload(data_io)
from data_io import export_stakeholder_outputs, export_perception_charts

# ── §5A — City-level stakeholder map ──────────────────────────────────────────
export_paths = export_stakeholder_outputs(
    stakeholder_map=stakeholder_map,
    cities_gdf=cities_gdf,
    workshop_prof=workshop_prof,
    output_dir=RESULTS_STAKE,
    study_year=config.PROJECT_STUDY_YEAR,
    figure_dpi=config.FIGURE_DPI,
)

# ── §7A — City-level perception figures ───────────────────────────────────────
perception_paths = export_perception_charts(
    perception_fig=perception_fig,
    donuts_fig=perception_donuts_fig,
    unique_resp=unique_resp,
    output_dir=RESULTS_STAKE,
    study_year=config.PROJECT_STUDY_YEAR,
    figure_dpi=config.FIGURE_DPI,
)

# ── §5B + §6 + §7B — Cluster figures ─────────────────────────────────────────
_figs_dir = RESULTS_STAKE / 'Figures'
_figs_dir.mkdir(parents=True, exist_ok=True)

for _fname, _fig in [
    ('cluster25_stakeholder_map',    fig_cluster_map),
    ('cluster25_stakeholder_donuts', fig_cluster_donuts),
    ('cluster25_perception_chart',   fig_prc_clust),
    ('cluster25_perception_donuts',  _prc_donuts),
]:
    for _ext in ['.svg', '.pdf', '.png']:
        _p = _figs_dir / f'{_fname}{_ext}'
        _fig.savefig(_p, bbox_inches='tight', dpi=config.FIGURE_DPI)
        print(f'  {_p.name}')

# ── Clean workshop participant data for NB03 §10 ──────────────────────────────
# Exports the fully cleaned workshop_df (NB02 filtering applied: off-target cities
# removed, absorbed city name remaps, multi-city attendance resolved).
# NB03 §10a loads this file instead of re-loading and re-filtering the raw survey.
_data_dir = RESULTS_STAKE / 'Data'
_data_dir.mkdir(parents=True, exist_ok=True)

_ws_cols = [
    'pid', 'matched_city_y',
    '61_temp_increase_10yrs',   # Q61 — perceived warming (NB03 §10 annual)
    '58_temp_peak_city',        # Q58 — peak season (NB03 §10 monthly)
    '60_diurnal_severity_heat', # Q60 — diurnal severity (NB02 §7 only)
    '56_recency_with_city',     # duration of residence in city
]
_ws_export = workshop_df[
    [c for c in _ws_cols if c in workshop_df.columns]
].drop_duplicates('pid')
_ws_path = _data_dir / 'workshop_attendees_clean.csv'
_ws_export.to_csv(_ws_path, index=False)
print(f'\n  workshop_attendees_clean.csv  ({len(_ws_export)} attendees, '
      f'{len(_ws_export.columns)} columns)  → {_ws_path.name}')

print("\nExport complete.")
for p in list(export_paths.values()) + list(perception_paths.values()):
    print(f"  {p}")

---
<a id="nb02-s09"></a>
## Section 9 — Summary & Next Steps

> ↑ [Back to Table of Contents](#nb02-toc)

---

## Next → Notebook 03 — Monthly & Annual LST Temporal Analysis

Open **`Notebook/03_temporal_analysis_months_annual.ipynb`**

| What NB03 does | Required inputs |
|----------------|-----------------|
| MODIS MYD11A1 three-zone LST extraction (2006–2025) | NB01 GEE assets (`Nigeria_AOI/{city_slug}`) |
| Temporal trend analysis — MK / Hamed-Rao / Sen slope / OLS R² | NB01 per-city `*_city_region_4326_v5.geojson` |
| Cluster25-based LST composites (§7a) | NB01 `city_cluster25_lookup.csv` |
| Cross-city heatmaps with `[C{N}]` cluster labels (§8) | NB01 `city_cluster25_lookup.csv` |
| Stakeholder perception validation (§10) | **NB02** survey outputs (Q61, Q60) |

> Before opening NB03, confirm:
> - [ ] All NB01 §10 GEE export tasks have completed (check GEE task manager)
> - [ ] `Results/01_aoi_delineation/city_cluster25_lookup.csv` exists
> - [ ] NB02 has run and `Results/02_stakeholder_analysis/` outputs are present (required for §10)